# IRWA Final Project — Part 3: Ranking & Filtering

Every method uses the same AND filter from Part 2 (`conjunctive_search`): a product is returned only
if it contains all the query terms. Only the order of the results changes.

**Run from the repository root or from this folder.** Required files: `data/processed_corpus.pkl`
(output of Part 1) and `project_progress/part_2/test_queries_to_label.csv`.

In [1]:
import sys, math
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "myapp").is_dir())
sys.path.insert(0, str(ROOT))

from myapp.core.preprocessing import preprocess_text
from myapp.search.algorithms import create_index, conjunctive_search
from myapp.search.ranking import create_ranking_data, search_ranked, rank_bm25, tfidf_weight
from myapp.search.evaluation import get_labels_and_scores, precision_at_k, avg_precision_at_k, ndcg_at_k

DATA = ROOT / "data"
PART2 = ROOT / "project_progress" / "part_2"
PART3 = ROOT / "project_progress" / "part_3"
pd.set_option("display.max_colwidth", 60)

In [2]:
corpus = pd.read_pickle(DATA / "processed_corpus.pkl")
index, tf, idf = create_index(corpus)
data = create_ranking_data(corpus, idf)
title = dict(zip(corpus.pid, corpus.title))
print(f"{len(corpus):,} products, average length {data['avg_length']:.1f} tokens")

28,080 products, average length 39.1 tokens


In [3]:
test_labels = pd.read_csv(PART2 / "test_queries_to_label.csv")
TEST_QUERIES = test_labels.drop_duplicates("query_id").set_index("query_id")["query"].to_dict()
TEST_QUERIES

{1: 'men cotton polo tshirt',
 2: 'women cotton kurta',
 3: 'men hooded sweatshirt',
 4: 'men running shoes',
 5: 'women blue denim jacket'}

## 1. TF-IDF + cosine similarity and BM25

Both are in `myapp/search/ranking.py` and use the formulas of the lectures.

**TF-IDF + cosine**

* weight of a term in a text: $w_{t} = (1 + \log_2 f_t) \cdot idf_t$
* score: $\cos(q, d) = \dfrac{\sum_t w_{t,q} \cdot w_{t,d}}{\lVert q \rVert \cdot \lVert d \rVert}$

$\lVert d \rVert$ is the length of the **whole** tf-idf vector of the product, with all its terms.
In Part 2 we only divided tf by the length of the count vector, so that was not an exact cosine.

**BM25**

$$RSV_d = \sum_{t \in q} idf_t \cdot \frac{(k_1 + 1)\, tf_{t,d}}{k_1\left((1 - b) + b \cdot \frac{L_d}{L_{ave}}\right) + tf_{t,d}}$$

* $L_d$ = number of tokens of the product, $L_{ave}$ = average over the corpus
* $k_1 = 1.2$: repeating a term helps less and less (saturation)
* $b = 0.75$: how much long products are penalised

`idf` is the one from Part 2, $\log(N/df)$. Using another log base multiplies every score by the
same number, so the ranking does not change.

### Check with the exercise of the sample midterm

Query "hardware components", $k_1 = 1$, $b = 0.75$, $\log_2$. Expected: $w$ = 0.323, 0.643 and 0.737;
BM25 $RSV_{d1} = 1.11$, $RSV_{d2} = 1.08$.

In [4]:
from collections import Counter

exam_idf = {"hardware": math.log2(5 / 4), "component": math.log2(5 / 3)}
exam_data = {
    "counts": {"d1": Counter({"hardware": 1, "component": 1}), "d2": Counter({"hardware": 2, "component": 1})},
    "doc_lengths": {"d1": 7, "d2": 10},
    "avg_length": 8,
}
weights = [tfidf_weight(1, exam_idf["hardware"]), tfidf_weight(2, exam_idf["hardware"]), tfidf_weight(1, exam_idf["component"])]
rsv = dict(rank_bm25(["hardware", "component"], ["d1", "d2"], exam_idf, exam_data, k1=1, b=0.75))

assert [round(w, 2) for w in weights] == [0.32, 0.64, 0.74]
assert round(rsv["d1"], 2) == 1.11 and round(rsv["d2"], 2) == 1.08
print("weights:", [round(w, 3) for w in weights], "| RSV:", {d: round(s, 2) for d, s in rsv.items()})

weights: [0.322, 0.644, 0.737] | RSV: {'d1': 1.11, 'd2': 1.08}


### Top 20 of our five queries

In [5]:
def top20(query, method):
    #results table [rank, pid, score, title, our label from Part 2 or NaN if not labelled]
    labels = test_labels[test_labels["query"] == query].set_index("pid")["label"]
    res = pd.DataFrame(search_ranked(query, index, idf, data, method=method, k=20), columns=["pid", "score"])
    res.insert(0, "rank", np.arange(1, len(res) + 1))
    res["title"] = res.pid.map(title)
    res["label"] = res.pid.map(labels)
    return res.round({"score": 4})

results = {(qid, m): top20(q, m) for qid, q in TEST_QUERIES.items() for m in ["tfidf", "bm25"]}

In [6]:
for qid, q in TEST_QUERIES.items():
    for method, name in [("tfidf", "TF-IDF + cosine"), ("bm25", "BM25")]:
        print(f"Query {qid}: {q} | {name}")
        display(results[(qid, method)])

Query 1: men cotton polo tshirt | TF-IDF + cosine


,rank,pid,score,title,label
0,1,TSHFUGZ9ZG5XN2FY,0.7847,Solid Men Polo Neck Blue T-Shirt,1.0
1,2,TSHFUGNWKWGPZ32S,0.7817,Solid Men Polo Neck Black T-Shirt,1.0
2,3,TSHFX3HD76TRJT3T,0.6706,Solid Men Polo Neck Grey T-Shirt,NaN
3,4,TSHFZZ2WHKH4EGKZ,0.6601,Solid Men Polo Neck Blue T-Shirt,NaN
4,5,TSHFUGKSZHBWZZ5E,0.6583,Solid Men Polo Neck Multicolor T-Shirt,1.0
5,6,TSHFDCH4EHXXBRRC,0.6527,Solid Men Polo Neck Black T-Shirt,NaN
6,7,TSHFZBYXXZVG4YG7,0.6527,Solid Men Polo Neck Black T-Shirt,NaN
7,8,TSHFUNN3HZKFZZDJ,0.6448,Printed Men Polo Neck Grey T-Shirt,NaN
8,9,TSHFP5Y5HYSMPNEV,0.6390,Printed Men Polo Neck Black T-Shirt,NaN
9,10,TSHEQFQ2FRKWAFZC,0.6365,Solid Men Polo Neck Black T-Shirt,NaN


Query 1: men cotton polo tshirt | BM25


,rank,pid,score,title,label
0,1,TSHFUNN3HZKFZZDJ,6.8448,Printed Men Polo Neck Grey T-Shirt,NaN
1,2,TSHFUNN36MB6BAPG,6.8012,Solid Men Polo Neck Yellow T-Shirt,NaN
2,3,TSHFUNN2YHM6XEHR,6.7582,Striped Men Polo Neck White T-Shirt,NaN
3,4,TSHFZKJHBGGUBASG,6.7582,Solid Men Polo Neck Grey T-Shirt,NaN
4,5,TSHFZKJHQ4GWPCDY,6.7582,Solid Men Polo Neck Grey T-Shirt,NaN
5,6,TSHFUGNWKWGPZ32S,6.7245,Solid Men Polo Neck Black T-Shirt,1.0
6,7,TSHFUGZ9ZG5XN2FY,6.7245,Solid Men Polo Neck Blue T-Shirt,1.0
7,8,TSHFZHPBVYPHHC2J,6.7158,Solid Men Polo Neck Pink T-Shirt,NaN
8,9,TSHFZHPBXMRMMUY4,6.7158,Solid Men Polo Neck Green T-Shirt,NaN
9,10,TSHFZKJHASFMNT2Y,6.7158,Solid Men Polo Neck Grey T-Shirt,NaN


Query 2: women cotton kurta | TF-IDF + cosine


,rank,pid,score,title,label
0,1,KTAFAKGUDJEB5DR7,0.5590,Women Solid Cotton Blend Straight Kurta (Black),NaN
1,2,KTAFH7NBG8PEX5GP,0.5186,"Women Solid Cotton Blend Straight Kurta (Maroon, Yellow)",NaN
2,3,KTAFMRFBSZ5QYCUW,0.5099,Women Solid Pure Cotton Straight Kurta (Black),NaN
3,4,KTAFAJFFHXVJHUMU,0.5094,Women Solid Cotton Blend Straight Kurta (Green),1.0
4,5,KTAFZM3FNPJS7QUK,0.5039,Women Solid Cotton Blend Straight Kurta (Dark Blue),NaN
5,6,KTAFZM3FPYWFMP7W,0.4987,Women Solid Cotton Blend Straight Kurta (Yellow),NaN
6,7,KTAFAJFFZYZTVNRA,0.4984,Women Solid Cotton Blend Straight Kurta (White),1.0
7,8,FABFWN82HVT5KAH2,0.4920,Cotton Printed Kurta Fabric (Unstitched),NaN
8,9,ETHFPDUMHA2BTSZH,0.4916,Women Kurta and Pyjama Set Pure Cotton,NaN
9,10,ETHFPDUMHFZMYHCX,0.4916,Women Kurta and Pyjama Set Pure Cotton,NaN


... (10 outputs left out)

Query 5: women blue denim jacket | TF-IDF + cosine


,rank,pid,score,title,label
0,1,JCKFZ33GHPEXHFUZ,0.9084,Full Sleeve Solid Women Jacket,1
1,2,JCKFWC5YGQENYT6B,0.7812,Full Sleeve Solid Women Denim Jacket,1
2,3,JCKFYHVRWMSWQJ2J,0.7587,Full Sleeve Solid Women Jacket,1
3,4,JCKFSHEZTYYPEMTJ,0.7408,Full Sleeve Solid Women Denim Jacket,1
4,5,JCKFVDPXFVQBGME7,0.7393,Full Sleeve Solid Women Denim Jacket,1
5,6,JCKFXNVMGBYGXKHN,0.7386,Full Sleeve Solid Women Denim Jacket,1
6,7,JCKFVEUDNF3J7CWV,0.7321,Full Sleeve Solid Women Jacket,1
7,8,JCKFVEUDBT4KKSXR,0.7196,Full Sleeve Solid Women Jacket,1
8,9,JCKFSJZVVFNUYTZW,0.6535,Full Sleeve Self Design Women Jacket,1
9,10,JCKF7KY3NQUGDHE2,0.6322,Full Sleeve Solid Women Denim Casual Jacket,1


Query 5: women blue denim jacket | BM25


,rank,pid,score,title,label
0,1,JCKFZ33GHPEXHFUZ,13.1808,Full Sleeve Solid Women Jacket,1
1,2,JCKFWC5YGQENYT6B,13.0658,Full Sleeve Solid Women Denim Jacket,1
2,3,JCKFSHEZTYYPEMTJ,12.8926,Full Sleeve Solid Women Denim Jacket,1
3,4,JCKFVDPXFVQBGME7,12.5812,Full Sleeve Solid Women Denim Jacket,1
4,5,JCKFXNVMGBYGXKHN,12.5812,Full Sleeve Solid Women Denim Jacket,1
5,6,JCKFVEUDNF3J7CWV,12.3904,Full Sleeve Solid Women Jacket,1
6,7,JCKFVEUDBT4KKSXR,12.2949,Full Sleeve Solid Women Jacket,1
7,8,JCKFV7V7ZQ3PUSFF,12.2859,Full Sleeve Washed Women Denim Riding Jacket,1
8,9,JCKF7KY3NQUGDHE2,12.2065,Full Sleeve Solid Women Denim Casual Jacket,1
9,10,JCKFWQE5VX2YHF9J,12.2010,Full Sleeve Animal Print Women Jacket,1


### How different are the two rankings?

* **overlap**: products that are in both top 20
* **median length**: median number of tokens of the top 20 products
* **labelled 1 / 0 / not labelled**: our Part 2 labels only cover the top 20 of the Part 2 ranking,
  so products that are new in the top 20 have no label

In [7]:
rows = []
for qid, q in TEST_QUERIES.items():
    n_matches = len(conjunctive_search(preprocess_text(q), index))
    tfidf_res, bm25_res = results[(qid, "tfidf")], results[(qid, "bm25")]
    row = {"query": q, "AND matches": n_matches,
           "overlap": len(set(tfidf_res.pid) & set(bm25_res.pid))}
    for name, res in [("tfidf", tfidf_res), ("bm25", bm25_res)]:
        row[f"{name} median length"] = int(np.median([data["doc_lengths"][p] for p in res.pid]))
        row[f"{name} labelled 1/0/none"] = f"{int((res.label == 1).sum())}/{int((res.label == 0).sum())}/{int(res.label.isna().sum())}"
    rows.append(row)
comparison = pd.DataFrame(rows, index=pd.Index(TEST_QUERIES.keys(), name="query_id"))
comparison

,query,AND matches,overlap,tfidf median length,tfidf labelled 1/0/none,bm25 median length,bm25 labelled 1/0/none
query_id,,,,,,,
1,men cotton polo tshirt,1213,4,21,3/0/17,26,6/0/14
2,women cotton kurta,358,3,23,2/0/18,24,3/0/17
3,men hooded sweatshirt,323,0,19,0/0/20,29,18/0/2
4,men running shoes,37,18,27,11/6/3,31,11/8/1
5,women blue denim jacket,27,20,27,13/7/0,27,13/7/0


* With few AND matches (queries 4 and 5) both methods return almost the same products.
* With many near-identical matches (queries 1-3) the top 20 is very different, and most of those
  products were never labelled in Part 2.
* Both methods move the pack of socks of "men running shoes" down, from rank 4 in Part 2 to rank 13
  (TF-IDF) and 11 (BM25): it is a long product (58 tokens) that repeats "shoe" 8 times.
* TF-IDF + cosine puts an unstitched kurta **fabric** at rank 8 of "women cotton kurta": a short text
  with all the query terms wins even if it is not the product the user wants.

### Why do they disagree? Example: "men hooded sweatshirt"

The two methods share **no** product in their top 20. Top 3 of each, with the length of the product
and how many times it contains each query term:

In [8]:
q = "men hooded sweatshirt"
terms = preprocess_text(q)
for method in ["tfidf", "bm25"]:
    for pid, score in search_ranked(q, index, idf, data, method=method, k=3):
        counts = {t: data["counts"][pid][t] for t in terms}
        print(f"{method:5} {score:.4f}  {title[pid]:38} length {data['doc_lengths'][pid]:3}  {counts}")

tfidf 0.8191  Full Sleeve Solid Men Sweatshirt       length  20  {'men': 1, 'hood': 2, 'sweatshirt': 2}
tfidf 0.8175  Full Sleeve Solid Men Sweatshirt       length  20  {'men': 1, 'hood': 2, 'sweatshirt': 2}
tfidf 0.8146  Full Sleeve Solid Men Sweatshirt       length  20  {'men': 1, 'hood': 2, 'sweatshirt': 2}
bm25  12.2004  Full Sleeve Printed Men Sweatshirt     length  32  {'men': 3, 'hood': 4, 'sweatshirt': 4}
bm25  12.2004  Full Sleeve Printed Men Sweatshirt     length  32  {'men': 3, 'hood': 4, 'sweatshirt': 4}
bm25  12.0873  Full Sleeve Printed Men Sweatshirt     length  34  {'men': 3, 'hood': 4, 'sweatshirt': 4}


* **TF-IDF + cosine** divides by the length of the whole product vector, so it prefers the **shortest**
  products, where the query terms are a big part of the text (20 tokens, each term once or twice).
* **BM25** rewards products that **repeat** the query terms more (here 3-4 times), and only penalises
  length a little: these products have 32 tokens, close to the average of 39.
* All 20 TF-IDF results also have `Hooded: Yes` in their details, so a different top 20 does not mean a
  worse one. Most of them were not labelled in Part 2.

### Effect of the BM25 parameters

Overlap between the default BM25 top 20 ($k_1 = 1.2$, $b = 0.75$) and other settings, for the query with
most matches (men cotton polo tshirt, 1,213 products):

In [9]:
q = "men cotton polo tshirt"
terms = preprocess_text(q)
docs = conjunctive_search(terms, index)
default = {pid for pid, _ in rank_bm25(terms, docs, idf, data)[:20]}
settings = [(1.2, 0), (1.2, 1), (2.0, 0.75), (100, 0.75)]
pd.DataFrame([{"k1": k1, "b": b, "overlap with default": len(default & {pid for pid, _ in rank_bm25(terms, docs, idf, data, k1=k1, b=b)[:20]})}
              for k1, b in settings])

,k1,b,overlap with default
0,1.2,0.00,0
1,1.2,1.00,16
2,2.0,0.75,19
3,100.0,0.75,15


* $b = 0$ (no length normalisation) gives a completely different top 20: in this corpus the length of
  the product decides a lot.
* A very large $k_1$ removes the saturation, so BM25 behaves more like plain tf and the top 20 changes again.

### Validation queries

The labelled products of `validation_labels.csv`, ranked with each method (same procedure as Part 2:
products the AND search does not return get score 0).

In [10]:
labels = pd.read_csv(DATA / "validation_labels.csv")
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}

rows = []
for method in ["tfidf", "bm25"]:
    for qid, q in VALIDATION_QUERIES.items():
        judged = labels[labels.query_id == qid]
        scores = dict(search_ranked(q, index, idf, data, method=method, k=None))
        y_true = judged["labels"].to_numpy()
        y_score = np.array([scores.get(pid, 0) for pid in judged.pid])
        rows.append({"method": method, "query_id": qid,
                     "P@10": precision_at_k(y_true, y_score, 10),
                     "AP@10": avg_precision_at_k(y_true, y_score, 10),
                     "NDCG@10": ndcg_at_k(y_true, y_score, 10)})
pd.DataFrame(rows).round(3)

,method,query_id,P@10,AP@10,NDCG@10
0,tfidf,1,1.0,1.000,1.000
1,tfidf,2,0.6,0.948,0.716
2,bm25,1,1.0,1.000,1.000
3,bm25,2,0.6,0.948,0.716


Both methods give the same values as the Part 2 ranking at K = 10: the 20 labelled products are
ordered almost in the same way. Since they are spread over the whole result list (Part 2), this set
cannot show the differences we see in the top 20 of our own queries.

In [11]:
#save both top 20 lists for the report
top20_all = pd.concat([res.assign(method=m, query_id=qid, query=TEST_QUERIES[qid]) for (qid, m), res in results.items()])
top20_all.to_csv(PART3 / "results_tfidf_bm25_top20.csv", index=False)
comparison.to_csv(PART3 / "results_tfidf_vs_bm25.csv")
print("Saved results_tfidf_bm25_top20.csv and results_tfidf_vs_bm25.csv")

Saved results_tfidf_bm25_top20.csv and results_tfidf_vs_bm25.csv


### Summary: TF-IDF + cosine vs BM25

| | TF-IDF + cosine | BM25 |
|---|---|---|
| repeated terms | grow with log, without limit | saturate ($k_1$) |
| length | divides by the length of the whole tf-idf vector, so short products win | soft penalty relative to the average length ($b$) |
| parameters | none | $k_1$ and $b$, must be tuned |
| when the AND leaves few products (Q4, Q5) | almost the same top 20 as BM25 | almost the same top 20 as TF-IDF |
| when it leaves many near-identical products (Q1–Q3) | very different top 20 | very different top 20 |